# Qwen-27B 实际同幅度随机阻断

保留原 32 个匹配四元组、原 L48/L49/L55 方向构造和所有原有条件。

[实验设置与运行说明](../../../docs/experiment_notes/04_section45_guided_repair/31b29d6afde9d032.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')

IMPLEMENTATION_SHA256 = '160372af558929c42134517d3ffc09527278d018135dd39a46f32eb6987dde7d'
IMPLEMENTATION_SHA256 = 'f45e97fe3369f2f0304c9ae1fca75dde0df9d5b411e6f77ed7b5bfdc33c368c7'


In [ ]:
%%cell_progress 1/15 Mount Drive
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/15 Install inference environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/15 Check GPU access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/15 Load shared validation utilities
"""Pure CPU utilities embedded in the follow-up notebooks; no model execution."""
import hashlib
import json
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations, product
import numpy as np
import pandas as pd


def plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=plain, allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_jsonl(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=plain, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)


def freeze(path, identity):
    identity = json.loads(json.dumps(identity, default=plain, sort_keys=True, allow_nan=False))
    if Path(path).is_file():
        assert read_json(path) == identity, f'Configuration changed; preserve the old run and use a new RUN_ID: {path}'
    else:
        save_json(path, identity)
    return digest(identity)


def cached_json(directory, key, identity, compute):
    path = Path(directory) / (digest(str(key))[:24] + '.json')
    expected = digest(identity)
    if path.is_file():
        old = read_json(path)
        assert old['key'] == str(key) and old['identity'] == expected, f'Incompatible cache: {path}'
        return old['result']
    value = compute()
    save_json(path, dict(key=str(key), identity=expected, result=value))
    return value


def bootstrap(values, key='', replicates=10000):
    x = np.asarray(values, dtype=float)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    rng = np.random.default_rng(int(digest(str(key))[:16], 16))
    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(1)
                            for i in range(0, replicates, 250)])
    return dict(n=len(x), mean=float(x.mean()), lower=float(np.quantile(draws, .025)),
                upper=float(np.quantile(draws, .975)),
                p_value=float((1 + (draws <= 0).sum()) / (replicates + 1)))


def bh(values):
    p = np.asarray(values, dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    order = np.argsort(p)
    q = np.empty_like(p)
    previous = 1.
    for rank in range(len(p)-1, -1, -1):
        i = order[rank]
        previous = min(previous, p[i]*len(p)/(rank+1))
        q[i] = previous
    return q


def paired(left, right, id_col, value):
    assert left[id_col].is_unique and right[id_col].is_unique
    assert set(left[id_col]) == set(right[id_col]), 'Different paired support'
    out = left[[id_col, value]].merge(right[[id_col, value]], on=id_col, validate='one_to_one', suffixes=('_l', '_r'))
    return out[value+'_l'].to_numpy(float) - out[value+'_r'].to_numpy(float)


def semantic_scores(record):
    if 'semantic_logprobs' in record:
        return {str(k): float(v) for k, v in record['semantic_logprobs'].items()}
    values = record['candidate_logprobs']
    if isinstance(values, dict):
        return {str(record['label_to_semantic'][k]): float(v) for k, v in values.items()}
    return dict(zip(map(str, record['candidate_semantics']), map(float, values)))


def correct_margin(record):
    values = semantic_scores(record)
    gold = str(record['gold_semantic'])
    assert gold in values and len(values) >= 2
    return values[gold] - max(v for k, v in values.items() if k != gold)


def random_unit(shape, key):
    r = np.random.default_rng(int(digest(key)[:16], 16)).normal(size=shape)
    return r / np.linalg.norm(r)


def coordinate_clamp(pre, untreated, axis, cast):
    h, base, u = [np.asarray(x, dtype=np.float64) for x in (pre, untreated, axis)]
    u = u / np.linalg.norm(u)
    a = float(np.sum((h-base)*u))
    return np.asarray(cast(h-a*u), dtype=np.float64), a


def native_norm_edit(pre, direction, norm, cast, tolerance=.02, steps=48):
    h = np.asarray(pre, dtype=np.float64)
    r = np.asarray(direction, dtype=np.float64)
    assert np.isfinite(h).all() and np.isfinite(r).all() and norm >= 0 and np.linalg.norm(r) > 0
    r /= np.linalg.norm(r)
    if norm == 0:
        return h.copy(), dict(actual_norm=0., target_norm=0., relative_error=0., matched=True)
    lo, hi = 0., max(float(norm), 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h+hi*r))-h) >= norm:
            break
        hi *= 2
    best, error = h.copy(), 1.
    for _ in range(steps):
        mid = (lo+hi)/2
        candidate = np.asarray(cast(h+mid*r), dtype=np.float64)
        actual = float(np.linalg.norm(candidate-h))
        current = abs(actual-norm)/norm
        if current < error:
            best, error = candidate.copy(), current
        if error <= tolerance:
            break
        if actual < norm:
            lo = mid
        else:
            hi = mid
    return best, dict(actual_norm=float(np.linalg.norm(best-h)), target_norm=float(norm),
                      relative_error=float(error), matched=bool(error <= tolerance))


def equal_removal(pre, untreated, axis, random_direction, cast, tolerance=.02):
    target, coefficient = coordinate_clamp(pre, untreated, axis, cast)
    target_norm = float(np.linalg.norm(target-np.asarray(pre)))
    sign = 1. if coefficient >= 0 else -1.
    post, audit = native_norm_edit(pre, -sign*np.asarray(random_direction), target_norm, cast, tolerance)
    audit.update(reference_coefficient=coefficient, reference_actual_norm=target_norm)
    return post, audit


def geometry_signature(stacks):
    a = tuple(sorted((str(s['side']), int(s['distance']), int(s['units'])) for s in stacks))
    b = tuple(sorted(('right' if side == 'left' else 'left', d, u) for side, d, u in a))
    return min(a, b)  # mirrors belong to the same physical family


def geometries_in(value):
    found = set()
    if isinstance(value, dict):
        stacks = value.get('stacks')
        if isinstance(stacks, list) and stacks and all(isinstance(s, dict) and {'side','distance','units'} <= set(s) for s in stacks):
            found.add(geometry_signature(stacks))
        # Some prior matched cohorts store geometry instead of stack records.
        geom = value.get('pair_geometry')
        if isinstance(geom, (list, tuple)) and len(geom) == 4 and all(isinstance(x, (int,float)) for x in geom):
            u1,d1,u2,d2 = map(int, geom)
            found.add(geometry_signature([dict(side='left',distance=d1,units=u1),dict(side='right',distance=d2,units=u2)]))
        if isinstance(geom, (list, tuple)) and len(geom)==2 and all(isinstance(x,(list,tuple)) and len(x)==2 for x in geom):
            (u1,d1),(u2,d2)=geom
            found.add(geometry_signature([dict(side='left',distance=int(d1),units=int(u1)),dict(side='right',distance=int(d2),units=int(u2))]))
        for child in value.values():
            found.update(geometries_in(child))
    elif isinstance(value, list):
        for child in value:
            found.update(geometries_in(child))
    return found


def physics(stacks):
    units = {s:sum(int(x['units']) for x in stacks if x['side']==s) for s in ('left','right')}
    dist = {s:sum(int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    torque = {s:sum(int(x['units'])*int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    out = lambda x:'left' if x>0 else 'right' if x<0 else 'balanced'
    net=torque['left']-torque['right']; w=units['left']-units['right']; d=dist['left']-dist['right']
    return dict(net_torque=net,outcome=out(net),weight_score=w,distance_score=d,
                normalized_margin=abs(net)/max(1,sum(torque.values())),
                left_units=units['left'],right_units=units['right'],left_torque=torque['left'],right_torque=torque['right'],
                strategy_predictions=dict(exact_torque=out(net),weight_only=out(w),distance_only=out(d)))


def exclusion_snapshot(project, followup_root, dataset_dir=None):
    lock=Path(followup_root)/'prior_geometry_snapshot.json'
    if lock.is_file():
        old=read_json(lock)
        for path, sha in old['files'].items():
            assert file_hash(Path(path))==sha, f'Prior input changed: {path}'
        return {tuple(tuple(x) for x in sig) for sig in old['signatures']}, old
    paths={Path(dataset_dir or project/'data/dataset_v1')/'scenes.jsonl'}
    for pattern in ('*manifest*.jsonl','*support*.jsonl','*geometries*.jsonl'):
        paths.update(p for p in (project/'results').rglob(pattern) if Path(followup_root) not in p.parents)
    excluded=set(); hashes={}
    for path in sorted(paths):
        assert path.is_file(), f'Missing required geometry input: {path}'
        excluded.update(geometries_in(read_jsonl(path)))
        hashes[str(path.resolve())]=file_hash(path)
    assert excluded, 'No prior physical geometry was indexed'
    old=dict(files=hashes,signatures=sorted(excluded),mirror_canonical=True)
    save_json(lock,old)
    return excluded,old


def inverse_families(n, seed, excluded):
    assert n%4==0
    pool=defaultdict(list)
    for dl in combinations(range(1,7),2):
        for dr in combinations(range(1,7),2):
            for units in product(range(1,6), repeat=4):
                stacks=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),dl+dr,units)]
                f=physics(stacks)
                pool[dl+dr].append((stacks,f))
    candidates=[]
    for distances,rows in pool.items():
        by_net=defaultdict(list)
        for stacks,f in rows:
            if f['outcome']==f['strategy_predictions']['weight_only'] and f['outcome']!='balanced':
                by_net[f['net_torque']].append((stacks,f))
        for stacks,f in rows:
            margin=f['normalized_margin']; difficulty='hard' if .08<=margin<.22 else 'moderate' if .22<=margin<.45 else None
            if (difficulty and f['outcome']!='balanced' and f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome'])
                    and f['strategy_predictions']['distance_only']==f['outcome'] and abs(f['distance_score'])>=2):
                sources=by_net.get(f['net_torque'],[])
                if sources:candidates.append((digest([seed,stacks]),difficulty,stacks,f,sources))
    candidates.sort(key=lambda r:r[0]); used=set(excluded); counts=Counter(); result=[]
    for _,difficulty,stacks,f,sources in candidates:
        stratum=(difficulty,f['outcome'])
        if counts[stratum]>=n//4 or geometry_signature(stacks) in used:continue
        valid=[(s,g) for s,g in sources if geometry_signature(s) not in used|{geometry_signature(stacks)}]
        if not valid:continue
        valid.sort(key=lambda x:(sum(abs(a['units']-b['units']) for a,b in zip(stacks,x[0])),digest([seed,x[0]])))
        src,sf=valid[0];fid='fixed_'+digest([seed,stacks,src])[:20]
        roles={name:dict(scene_id=fid+'__'+name,image_filename=fid+'__'+name+'.png',stacks=s,features=g)
               for name,s,g in [('base',stacks,f),('aligned_source',src,sf)]}
        result.append(dict(family_id=fid,remapping_split='new_fixed_confirmation',roles=roles,margin_bin=difficulty))
        used.update([geometry_signature(stacks),geometry_signature(src)]);counts[stratum]+=1
        if len(result)==n:break
    assert len(result)==n and set(counts.values())=={n//4}, (len(result),counts)
    return sorted(result,key=lambda r:r['family_id'])


def repair_families(n, seed, excluded):
    assert n%4==0
    candidates=defaultdict(list)
    for ul,dl,ur,dr in product(range(1,9),range(1,7),range(1,9),range(1,7)):
        s=[dict(side='left',units=ul,distance=dl),dict(side='right',units=ur,distance=dr)]
        f=physics(s)
        if f['net_torque']==0:continue
        if ul==ur and dl!=dr:kind='equal_weight_distance'
        elif f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome']):kind='weight_distance_conflict'
        elif f['strategy_predictions']['weight_only']==f['outcome'] and f['strategy_predictions']['distance_only']==f['outcome']:kind='aligned_cues'
        else:continue
        candidates[kind].append(s)
    for units in product(range(1,6),repeat=4):
        for distances in [((1,3),(2,5)),((2,5),(1,4)),((1,6),(3,5)),((2,4),(3,6))]:
            s=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),distances[0]+distances[1],units)]
            if physics(s)['net_torque']!=0:candidates['multi_stack_sum'].append(s)
    used=set(excluded); result=[]
    for kind in ('equal_weight_distance','weight_distance_conflict','multi_stack_sum','aligned_cues'):
        rows=sorted(candidates[kind],key=lambda s:digest([seed,kind,s])); count=0
        for stacks in rows:
            sig=geometry_signature(stacks)
            if sig in used:continue
            fid='repair_'+digest([seed,kind,sig])[:20];scenes=[]
            for mirror in (False,True):
                current=[dict(s,side=('right' if s['side']=='left' else 'left')) if mirror else dict(s) for s in stacks]
                for style in ('standard','alternate'):
                    sid=f'{fid}__m{int(mirror)}__{style}'
                    scenes.append(dict(scene_id=sid,image_filename=sid+'.png',stacks=current,features=physics(current),style=style,mirror=mirror))
            result.append(dict(family_id=fid,stratum=kind,scenes=scenes));used.add(sig);count+=1
            if count==n//4:break
        assert count==n//4,(kind,count,'Insufficient unused geometries; do not silently change task ranges')
    return sorted(result,key=lambda r:r['family_id'])


In [ ]:
%%cell_progress 5/15 Freeze original route and new controls
RUN_ID='main'
RANDOM_SEEDS=(0,1,2,3)
cell_stage('Freezing new-data generation, prior mechanism, conditions, and decision gates')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
UPSTREAM_LAYER = 48
UPSTREAM_METHOD = 'pls4_prototype'
SEMANTIC_LAYER = 49
LABEL_LAYER = 55
CAPTURE_LAYERS = (UPSTREAM_LAYER, SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20261229
EXPANSION_SEED = 20261124
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
QUESTION_KINDS = ('outcome',)
COHORTS = ('both_correct', 'right_error')
SIDES = ('left', 'right')
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
PATCH_CONDITION_BATCH_SIZE = 1
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
CLAMP_PROJECTION_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02
BASELINE_REPLAY_MIN_CONCORDANCE = 0.90
BASELINE_REPLAY_MIN_CELL_CONCORDANCE = 0.80

GEOMETRIES_PER_WAVE = 48
MAXIMUM_WAVES = 3
MAXIMUM_GEOMETRIES = GEOMETRIES_PER_WAVE * MAXIMUM_WAVES
MINIMUM_MATCHED_UNITS = 20
TARGET_MATCHED_UNITS = 32
EARLY_STOP_EXACT_GEOMETRIES = 40
MINIMUM_STYLE_FAMILIES = 4
STYLE_BALANCE_TOLERANCES = (0, 1, 2, 3, 4)

PHYSICAL_FEATURES = (
    'units_1', 'distance_1', 'units_2', 'distance_2',
    'total_units', 'unit_difference', 'total_distance',
    'distance_difference', 'smaller_side_torque',
    'larger_side_torque', 'absolute_net_torque',
)
SELF_CONTROL_CONDITIONS = (
    'self_block_l49', 'self_random_block_l49',
    'self_block_l55', 'self_random_block_l55',
)
CORE_INTERVENTION_CONDITIONS = (
    'upstream_repair', 'upstream_repair_block_l49',
    'upstream_repair_random_block_l49',
    'upstream_repair_block_l55',
    'upstream_repair_random_block_l55',
)
CORE_CONDITIONS = SELF_CONTROL_CONDITIONS + CORE_INTERVENTION_CONDITIONS
RIGHT_ONLY_CONDITIONS = (
    'l49_repair', 'l49_repair_block_l55',
    'l49_repair_random_block_l55', 'l55_repair',
    'l55_bypass_after_l49_block',
)
ALL_CONDITIONS = CORE_CONDITIONS + RIGHT_ONLY_CONDITIONS

# These six palettes and two layouts per palette were not used by the parent cohort.
STYLE_SPECS = (
    {'style_id': 'cobalt_low', 'style_family': 'cobalt',
     'background': [239, 245, 255], 'ink': [24, 39, 71],
     'block_fill': [37, 99, 235], 'block_size': 26, 'beam_y': 276},
    {'style_id': 'cobalt_high', 'style_family': 'cobalt',
     'background': [239, 245, 255], 'ink': [24, 39, 71],
     'block_fill': [37, 99, 235], 'block_size': 36, 'beam_y': 300},
    {'style_id': 'coral_low', 'style_family': 'coral',
     'background': [255, 244, 241], 'ink': [78, 37, 31],
     'block_fill': [225, 88, 72], 'block_size': 27, 'beam_y': 280},
    {'style_id': 'coral_high', 'style_family': 'coral',
     'background': [255, 244, 241], 'ink': [78, 37, 31],
     'block_fill': [225, 88, 72], 'block_size': 35, 'beam_y': 297},
    {'style_id': 'forest_low', 'style_family': 'forest',
     'background': [240, 249, 243], 'ink': [28, 59, 40],
     'block_fill': [46, 139, 87], 'block_size': 28, 'beam_y': 274},
    {'style_id': 'forest_high', 'style_family': 'forest',
     'background': [240, 249, 243], 'ink': [28, 59, 40],
     'block_fill': [46, 139, 87], 'block_size': 34, 'beam_y': 302},
    {'style_id': 'lilac_low', 'style_family': 'lilac',
     'background': [249, 244, 255], 'ink': [57, 39, 75],
     'block_fill': [142, 91, 191], 'block_size': 25, 'beam_y': 286},
    {'style_id': 'lilac_high', 'style_family': 'lilac',
     'background': [249, 244, 255], 'ink': [57, 39, 75],
     'block_fill': [142, 91, 191], 'block_size': 37, 'beam_y': 294},
    {'style_id': 'sand_low', 'style_family': 'sand',
     'background': [252, 248, 234], 'ink': [67, 53, 29],
     'block_fill': [194, 143, 45], 'block_size': 29, 'beam_y': 278},
    {'style_id': 'sand_high', 'style_family': 'sand',
     'background': [252, 248, 234], 'ink': [67, 53, 29],
     'block_fill': [194, 143, 45], 'block_size': 33, 'beam_y': 304},
    {'style_id': 'charcoal_low', 'style_family': 'charcoal',
     'background': [245, 245, 245], 'ink': [35, 35, 35],
     'block_fill': [91, 91, 91], 'block_size': 26, 'beam_y': 288},
    {'style_id': 'charcoal_high', 'style_family': 'charcoal',
     'background': [245, 245, 245], 'ink': [35, 35, 35],
     'block_fill': [91, 91, 91], 'block_size': 36, 'beam_y': 292},
)
STYLE_BY_ID = {row['style_id']: row for row in STYLE_SPECS}

PREVIOUS_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_exactphysics_sequential_handoff_blocking_v1'
)
PREVIOUS_RUN_PATH = PREVIOUS_ROOT / 'RUN_COMPLETE.json'
PREVIOUS_PROTOCOL_PATH = PREVIOUS_ROOT / 'frozen_protocol.json'
PREVIOUS_SCENE_PATH = PREVIOUS_ROOT / 'frozen_scene_manifest.jsonl'
PREVIOUS_DECISION_PATH = (
    PREVIOUS_ROOT / 'analysis' / 'sequential_handoff_decision.json'
)
PARENT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_nettorque_semantic_handoff_mediation_v1'
)
PARENT_ROUTE_PATH = PARENT_ROOT / 'selected_upstream_route.json'
PARENT_DIRECTION_PATH = PARENT_ROOT / 'direction_library.json'
AUDIT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_error_conditioned_physical_state_audit_v1'
)
AUDIT_OUTCOME_MANIFEST_PATH = AUDIT_ROOT / 'outcome_prompt_manifest.jsonl'
AUDIT_OUTCOME_SHARD_DIR = AUDIT_ROOT / 'outcome_capture_shards'
DEPTH_SUPPORT_PATH = (
    PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
    / 'depth_support.jsonl'
)
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'cross_model_followup_v1' / 'equal_blocking' / RUN_ID
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
GEOMETRY_PATH = OUTPUT_ROOT / 'candidate_geometries.jsonl'
CANDIDATE_PATH = OUTPUT_ROOT / 'rendered_candidates.jsonl'
SCREEN_SHARD_DIR = OUTPUT_ROOT / 'screen_shards'
SCREEN_RECORDS_PATH = OUTPUT_ROOT / 'screen_records.jsonl'
PAIR_TABLE_PATH = OUTPUT_ROOT / 'pair_behavior_table.csv'
EDGE_PATH = OUTPUT_ROOT / 'exact_geometry_edges.csv'
MATCH_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_match_manifest.jsonl'
SCENE_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_scene_manifest.jsonl'
SELECTED_IMAGE_DIR = OUTPUT_ROOT / 'selected_images'
MATCH_AUDIT_PATH = OUTPUT_ROOT / 'exact_matching_audit.json'
BASELINE_PATH = OUTPUT_ROOT / 'baseline_mapping_replay.jsonl'
DIRECTION_LIBRARY_PATH = OUTPUT_ROOT / 'direction_library.json'
SHARD_DIR = OUTPUT_ROOT / 'path_blocking_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for directory in (
    OUTPUT_ROOT, SCREEN_SHARD_DIR, SELECTED_IMAGE_DIR,
    SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

RUNTIME_ROOT = Path('/content/visualtorque_sequential_independent_v1')
RUNTIME_IMAGE_DIR = RUNTIME_ROOT / 'images'
RUNTIME_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    temporary.write_text(json.dumps(
        value, indent=2, ensure_ascii=False, default=json_default
    ) + '\n', encoding='utf-8')
    temporary.replace(path)

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(path)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

required_inputs = (
    PREVIOUS_RUN_PATH, PREVIOUS_PROTOCOL_PATH, PREVIOUS_SCENE_PATH,
    PREVIOUS_DECISION_PATH, PARENT_ROUTE_PATH, PARENT_DIRECTION_PATH,
    AUDIT_OUTCOME_MANIFEST_PATH, AUDIT_OUTCOME_SHARD_DIR,
    DEPTH_SUPPORT_PATH, INVERSE_SCENE_DIR,
)
for required in required_inputs:
    assert Path(required).exists(), f'Missing prerequisite: {required}'

previous_run = json.loads(PREVIOUS_RUN_PATH.read_text(encoding='utf-8'))
assert previous_run['primary_decision'] == (
    'ordered_handoff_mediation_supported_on_frozen_exactphysics_cohort'
)
previous_protocol = json.loads(
    PREVIOUS_PROTOCOL_PATH.read_text(encoding='utf-8')
)['identity']
assert previous_protocol['route'] == {
    'upstream': UPSTREAM_LAYER, 'semantic': SEMANTIC_LAYER,
    'response_code': LABEL_LAYER, 'method': UPSTREAM_METHOD,
}
MATCHED_UPSTREAM_ALPHA = float(previous_protocol['matched_upstream_alpha'])
parent_route = json.loads(PARENT_ROUTE_PATH.read_text(encoding='utf-8'))
parent_direction = json.loads(PARENT_DIRECTION_PATH.read_text(encoding='utf-8'))
PARENT_SEMANTIC_NORM = float(parent_direction['semantic_norm'])
assert parent_route['selected_layer'] == UPSTREAM_LAYER
assert parent_route['selected_method'] == UPSTREAM_METHOD


CORE_CONDITIONS=CORE_CONDITIONS+tuple(f'upstream_equal_l49_seed{s}' for s in RANDOM_SEEDS)+tuple(f'upstream_equal_l55_seed{s}' for s in RANDOM_SEEDS)
RIGHT_ONLY_CONDITIONS=RIGHT_ONLY_CONDITIONS+tuple(f'l49_equal_l55_seed{s}' for s in RANDOM_SEEDS)
ALL_CONDITIONS=CORE_CONDITIONS+RIGHT_ONLY_CONDITIONS
protocol_identity=dict(schema='actual-equal-removal-1.0',implementation=IMPLEMENTATION_SHA256,model_id=MODEL_ID,revision=MODEL_REVISION,
    prior_cohort='qwen27b_sequential_handoff_independent_replication_v1',layers=list(CAPTURE_LAYERS),
    dose=MATCHED_UPSTREAM_ALPHA,seeds=list(RANDOM_SEEDS),conditions=list(ALL_CONDITIONS),dtype=MODEL_DTYPE,batch_size=1,
    norm_tolerance=NORM_RELATIVE_TOLERANCE,primary_paths=['upstream_l49','upstream_l55','l49_l55'],new_family_replication=False)
freeze(PROTOCOL_LOCK_PATH,protocol_identity)
print('Output:',OUTPUT_ROOT)


In [ ]:
%%cell_progress 6/15 Load the existing cohort without repeating behavior selection
from visual_torque_toolkit.behavior.data import BehaviorExample
from visual_torque_toolkit.behavior import PromptCondition
import shutil
OLD_ROOT=PROJECT_ROOT/'results/qwen27b_sequential_handoff_independent_replication_v1'
for name in ('frozen_scene_manifest.jsonl','frozen_match_manifest.jsonl','frozen_protocol.json','RUN_COMPLETE.json'):
    assert (OLD_ROOT/name).is_file(),OLD_ROOT/name
scene_manifest=read_jsonl(OLD_ROOT/'frozen_scene_manifest.jsonl')
match_manifest=read_jsonl(OLD_ROOT/'frozen_match_manifest.jsonl')
MATCHED_UNITS=len(match_manifest)
assert MATCHED_UNITS==32 and len(scene_manifest)==4*MATCHED_UNITS
assert Counter((r['cohort'],r['side']) for r in scene_manifest)=={(c,s):32 for c in COHORTS for s in SIDES}
RUNTIME_IMAGE_DIR.mkdir(parents=True,exist_ok=True)
outcome_example_by_scene={};image_hashes={}
for row in scene_manifest:
    src=OLD_ROOT/'selected_images'/Path(row['image_path']).name;assert src.is_file(),src
    dst=RUNTIME_IMAGE_DIR/src.name
    if not dst.is_file() or file_hash(dst)!=file_hash(src):shutil.copy2(src,dst)
    image_hashes[str(src)]=file_hash(src)
    semantics=('left','balanced','right');gold=row['exact_semantic']
    outcome_example_by_scene[row['scene_id']]=BehaviorExample(example_id=row['scene_id'],question_id=row['scene_id'],
        scene_id=row['scene_id'],condition=PromptCondition.IMAGE_RULE,question_kind='outcome',variant='base',split='historical_control_extension',
        problem_type=row['diagnostic_cell'],outcome=gold,net_torque=int(row['net_torque']),prompt='replaced by original target prompt',
        image_path=dst,labels=LABELS,option_text=semantics,option_semantics=semantics,gold_label=LABELS[semantics.index(gold)],gold_semantic=gold)
direction_support=read_jsonl(DEPTH_SUPPORT_PATH)
DIRECTION_RUNTIME_DIR=RUNTIME_ROOT/'direction_images';DIRECTION_RUNTIME_DIR.mkdir(exist_ok=True)
for family in direction_support:
    for role in family['roles'].values():
        src=INVERSE_SCENE_DIR/role['image_filename'];dst=DIRECTION_RUNTIME_DIR/src.name;assert src.is_file(),src
        image_hashes[str(src)]=file_hash(src)
        if not dst.is_file() or file_hash(dst)!=file_hash(src):shutil.copy2(src,dst)
write_jsonl_atomic(SCENE_MANIFEST_PATH,scene_manifest)
write_jsonl_atomic(MATCH_MANIFEST_PATH,match_manifest)
print('Historical cohort:',MATCHED_UNITS,'families. No re-selection and no new-family claim.')


In [ ]:
%%cell_progress 7/15 Load original pinned model
cell_stage(f'Loading {MODEL_KEY} weights for the matched error-specific handoff test')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 8/15 Define original prompts
cell_stage('Separating physical sides from A/B/C response codes')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['pair_instance_id']), seed=EXPANSION_SEED
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"exactphysics_direction:{row['family_id']}:{name}",
        question_id=f"exactphysics_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='exactphysics_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"independent_exactphysics_target:{row['scene_id']}:{variant}",
            question_id=f"independent_exactphysics_target_{row['scene_id']}_{variant}",
            variant=f'independent_exactphysics_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')


In [ ]:
%%cell_progress 9/15 Define original route operations
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

def vector_projection(delta, direction):
    unit = unit_direction(direction).reshape(-1)
    return float(torch.dot(delta.float().reshape(-1), unit))

def forward_add_and_capture(example, layer, payload_selected, capture_layers):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    handles = []
    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        assert int(current.sum()) == int(payload.shape[0]) == 1
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    handles.append(DECODER_LAYERS[layer].register_forward_hook(intervention_hook))
    for capture_layer in capture_layers:
        assert capture_layer > layer, (layer, capture_layer)
        def capture_hook(_module, _inputs, output, capture_layer=capture_layer):
            captured[capture_layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()[0]
        handles.append(
            DECODER_LAYERS[capture_layer].register_forward_hook(capture_hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    return record, {
        capture_layer: tensor[mask] for capture_layer, tensor in captured.items()
    }, int(mask.sum())

print('Downstream state-capture intervention helper ready.')

def clamp_selected_coordinate(selected, untreated, axis):
    untreated = untreated.to(device=selected.device, dtype=selected.dtype)
    axis_float = axis.to(device=selected.device, dtype=torch.float32)
    axis_float = axis_float / torch.linalg.vector_norm(axis_float)
    displacement = selected.float() - untreated.float()
    coefficient = torch.sum(displacement * axis_float, dim=-1, keepdim=True)
    clamped = selected.float() - coefficient * axis_float
    return clamped.to(selected.dtype), float(coefficient.detach().cpu().item())

def forward_path_condition(example, operations, untreated_selected, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured, removed = {}, {}
    handles = []

    def make_hook(layer):
        operation = operations.get(layer)
        def hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            current = mask.to(hidden.device)
            selected = hidden[0, current]
            changed = False
            if operation is not None:
                action = operation['action']
                if action == 'add':
                    payload = operation['payload'].to(
                        device=hidden.device, dtype=hidden.dtype
                    )
                    selected = selected + payload
                    changed = True
                elif action == 'clamp_axis':
                    selected, coefficient = clamp_selected_coordinate(
                        selected, operation['untreated'], operation['axis']
                    )
                    removed[layer] = coefficient
                    changed = True
                else:
                    raise ValueError(action)
            captured[layer] = selected.detach().cpu().clone()
            if not changed:
                return None
            modified = hidden.clone()
            modified[0, current] = selected
            return replace_output_hidden(output, modified)
        return hook

    active_layers = sorted(set(operations) | {SEMANTIC_LAYER, LABEL_LAYER})
    for layer in active_layers:
        handles.append(DECODER_LAYERS[layer].register_forward_hook(make_hook(layer)))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    record = score_records(prepared, output.logits)[0]
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['removed_projection_l49'] = float(removed.get(SEMANTIC_LAYER, 0.0))
    record['removed_projection_l55'] = float(removed.get(LABEL_LAYER, 0.0))
    record['l48_payload_norm'] = float(
        selected_norm(operations[UPSTREAM_LAYER]['payload'])
        if UPSTREAM_LAYER in operations else 0.0
    )
    record['l49_axis_norm'] = float(
        selected_norm(operations[SEMANTIC_LAYER]['axis'])
        if (SEMANTIC_LAYER in operations
            and operations[SEMANTIC_LAYER]['action'] == 'clamp_axis') else 0.0
    )
    record['l55_axis_norm'] = float(
        selected_norm(operations[LABEL_LAYER]['axis'])
        if (LABEL_LAYER in operations
            and operations[LABEL_LAYER]['action'] == 'clamp_axis') else 0.0
    )
    return record, captured

def build_path_operations(row, variant, mapping, untreated_selected):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    exact_label = label_for(mapping, exact)
    wrong_label = label_for(mapping, wrong)
    upstream_payload = (
        toward_exact_sign * UPSTREAM_DIRECTION * MATCHED_UPSTREAM_ALPHA
    )
    semantic_payload = toward_exact_sign * SEMANTIC_DIRECTION
    code_payload = CODE_DIRECTIONS[(wrong_label, exact_label)]
    random_semantic = matched_random(
        semantic_payload,
        ('sequential-block', row['scene_id'], variant, 'l49'),
    )
    random_code = matched_random(
        code_payload,
        ('sequential-block', row['scene_id'], variant, 'l55'),
    )

    add_upstream = {'action': 'add', 'payload': upstream_payload}
    add_semantic = {'action': 'add', 'payload': semantic_payload}
    add_code = {'action': 'add', 'payload': code_payload}
    block_l49 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[SEMANTIC_LAYER],
        'axis': semantic_payload,
    }
    random_block_l49 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[SEMANTIC_LAYER],
        'axis': random_semantic,
    }
    block_l55 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[LABEL_LAYER],
        'axis': code_payload,
    }
    random_block_l55 = {
        'action': 'clamp_axis',
        'untreated': untreated_selected[LABEL_LAYER],
        'axis': random_code,
    }
    operations = {
        'self_block_l49': {SEMANTIC_LAYER: block_l49},
        'self_random_block_l49': {SEMANTIC_LAYER: random_block_l49},
        'self_block_l55': {LABEL_LAYER: block_l55},
        'self_random_block_l55': {LABEL_LAYER: random_block_l55},
        'upstream_repair': {UPSTREAM_LAYER: add_upstream},
        'upstream_repair_block_l49': {
            UPSTREAM_LAYER: add_upstream, SEMANTIC_LAYER: block_l49,
        },
        'upstream_repair_random_block_l49': {
            UPSTREAM_LAYER: add_upstream, SEMANTIC_LAYER: random_block_l49,
        },
        'upstream_repair_block_l55': {
            UPSTREAM_LAYER: add_upstream, LABEL_LAYER: block_l55,
        },
        'upstream_repair_random_block_l55': {
            UPSTREAM_LAYER: add_upstream, LABEL_LAYER: random_block_l55,
        },
        'l49_repair': {SEMANTIC_LAYER: add_semantic},
        'l49_repair_block_l55': {
            SEMANTIC_LAYER: add_semantic, LABEL_LAYER: block_l55,
        },
        'l49_repair_random_block_l55': {
            SEMANTIC_LAYER: add_semantic, LABEL_LAYER: random_block_l55,
        },
        'l55_repair': {LABEL_LAYER: add_code},
        'l55_bypass_after_l49_block': {
            UPSTREAM_LAYER: add_upstream,
            SEMANTIC_LAYER: block_l49,
            LABEL_LAYER: add_code,
        },
    }
    axes = {
        'semantic': semantic_payload, 'semantic_random': random_semantic,
        'code': code_payload, 'code_random': random_code,
    }
    return operations, axes

print('Sequential add-and-clamp path helper ready.')


In [ ]:
%%cell_progress 10/15 Extend the original operations with four actual-magnitude matched random removals
legacy_build_path_operations=build_path_operations
def build_path_operations(row,variant,mapping,untreated_selected):
    operations,axes=legacy_build_path_operations(row,variant,mapping,untreated_selected)
    for seed in RANDOM_SEEDS:
        for stem,parent,layer in [('upstream_equal_l49','upstream_repair_block_l49',SEMANTIC_LAYER),
                                  ('upstream_equal_l55','upstream_repair_block_l55',LABEL_LAYER),
                                  ('l49_equal_l55','l49_repair_block_l55',LABEL_LAYER)]:
            ops=dict(operations[parent]);old=ops[layer]
            ops[layer]=dict(old,action='equal_remove',random_key=[SEED,row['scene_id'],variant,stem,seed])
            operations[f'{stem}_seed{seed}']=ops
    return operations,axes

def forward_path_condition(example,operations,untreated_selected,metadata):
    prepared=prepare_examples([example]);mask=final_mask(prepared)
    captured={};removed={};audits={};handles=[]
    def make_hook(layer):
        operation=operations.get(layer)
        def hook(_module,_inputs,output):
            hidden=output_hidden_tensor(output);m=mask.to(hidden.device);pre=hidden[0,m];post=pre
            assert int(m.sum())==1 and hidden.dtype==torch.bfloat16
            if operation is not None:
                action=operation['action']
                if action=='add':
                    payload=operation['payload'].to(device=pre.device,dtype=pre.dtype)
                    post=pre+payload;audits[layer]=dict(action=action,matched=True,requested_norm=float(payload.float().norm()))
                elif action in ('clamp_axis','equal_remove'):
                    reference,a=clamp_selected_coordinate(pre,operation['untreated'],operation['axis'])
                    removed[layer]=a;reference_norm=float((reference.float()-pre.float()).norm())
                    if action=='clamp_axis':
                        post=reference;audits[layer]=dict(action=action,matched=True,reference_actual_norm=reference_norm)
                    else:
                        h=pre.detach().float().cpu().numpy()
                        direction=-np.sign(a if a!=0 else 1)*random_unit(h.shape,operation['random_key'])
                        cast=lambda x:torch.as_tensor(np.asarray(x).copy(),dtype=pre.dtype).float().numpy()
                        value,audit=native_norm_edit(h,direction,reference_norm,cast,tolerance=NORM_RELATIVE_TOLERANCE)
                        post=torch.as_tensor(value,device=pre.device,dtype=pre.dtype)
                        audits[layer]=dict(action=action,reference_actual_norm=reference_norm,**audit)
                else:raise ValueError(action)
                audits.setdefault(layer,dict(action=action,matched=True))
                audits[layer]['actual_norm']=float((post.float()-pre.float()).norm())
                audits[layer]['hidden_norm']=float(pre.float().norm())
                changed=hidden.clone();changed[0,m]=post
            captured[layer]=post.detach().cpu().clone()
            return replace_output_hidden(output,changed) if operation is not None else None
        return hook
    for layer in sorted(set(operations)|{SEMANTIC_LAYER,LABEL_LAYER}):
        handles.append(DECODER_LAYERS[layer].register_forward_hook(make_hook(layer)))
    try:
        with torch.inference_mode():out=adapter.model(**prepared['inputs'],use_cache=False)
    finally:
        for h in handles:h.remove()
    record=score_records(prepared,out.logits)[0]
    record.update(metadata,patched_tokens=int(mask.sum()),write_audits={str(k):v for k,v in audits.items()},
        norm_valid=all(v.get('matched',True) for v in audits.values()),
        removed_projection_l49=float(removed.get(SEMANTIC_LAYER,0.)),removed_projection_l55=float(removed.get(LABEL_LAYER,0.)))
    return record,captured


In [ ]:
%%cell_progress 11/15 Reconstruct and check the frozen directions
cell_stage('Reconstructing the parent route without new scene outcomes')
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

left_units, left_raw_norms = [], []
code_units, code_raw_norms = defaultdict(list), []
for row in tqdm(direction_support, desc='L49/L55 directions', unit='family'):
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(
        examples, (SEMANTIC_LAYER, LABEL_LAYER)
    )
    for task in DONOR_TASKS:
        base_key, source_key = f'{task}__base', f'{task}__aligned_source'
        base_semantic = task_semantic(row['roles']['base']['features'], task)
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], task
        )
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = semantic_delta if source_semantic == 'left' else -semantic_delta
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))
        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        transition = (
            label_for(mapping, base_semantic), label_for(mapping, source_semantic)
        )
        code_units[transition].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    ) for transition, values in code_units.items()
}

audit_manifest = pd.DataFrame(read_jsonl(AUDIT_OUTCOME_MANIFEST_PATH))
hidden_rows, captured_ids = [], []
for path in tqdm(
    sorted(AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')),
    desc='Loading prior L48 states', unit='shard',
):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    hidden_rows.append(payload['hidden'][str(UPSTREAM_LAYER)].float())
    captured_ids.extend(str(value) for value in payload['scene_ids'])
assert audit_manifest['scene_id'].astype(str).tolist() == captured_ids
X = torch.cat(hidden_rows, dim=0).numpy()
y = audit_manifest['net_torque'].to_numpy(dtype=float)
fit = audit_manifest['split'].isin(['probe_train', 'probe_dev']).to_numpy()
scaler = StandardScaler().fit(X[fit])
X_fit, y_fit = scaler.transform(X[fit]), y[fit]
pls = PLSRegression(n_components=4, scale=False, max_iter=1000)
pls.fit(X_fit, y_fit.reshape(-1, 1))
scores = pls.transform(X_fit)
latent_delta = scores[y_fit > 0].mean(axis=0) - scores[y_fit < 0].mean(axis=0)
score_centered = scores - scores.mean(axis=0, keepdims=True)
X_centered = X_fit - X_fit.mean(axis=0, keepdims=True)
decoder = np.linalg.lstsq(score_centered, X_centered, rcond=1e-5)[0]
raw_delta = (latent_delta @ decoder) * scaler.scale_
UPSTREAM_DIRECTION = torch.from_numpy(
    (raw_delta / 2).astype('float32')
).reshape(1, -1)

parent_upstream = next(
    row for row in parent_direction['upstream']
    if int(row['layer']) == UPSTREAM_LAYER and row['method'] == UPSTREAM_METHOD
)
write_json_atomic(ANALYSIS_DIR/'direction_reconstruction_audit.json',dict(
    current_upstream_norm=selected_norm(UPSTREAM_DIRECTION),archived_upstream_norm=float(parent_upstream['norm']),
    current_semantic_norm=SEMANTIC_STEERING_NORM,archived_semantic_norm=PARENT_SEMANTIC_NORM,
    current_code_norm=CODE_STEERING_NORM,archived_code_norm=float(parent_direction['code_norm'])))
assert abs(selected_norm(UPSTREAM_DIRECTION) - float(parent_upstream['norm'])) < 1e-4, 'Frozen upstream reconstruction differs; inspect direction_reconstruction_audit.json' 
assert abs(SEMANTIC_STEERING_NORM - PARENT_SEMANTIC_NORM) < 1e-4
direction_metadata = {
    'schema_version': 'fixed-mediator-matched-directions-1.0',
    'upstream_layer': UPSTREAM_LAYER, 'upstream_method': UPSTREAM_METHOD,
    'upstream_norm': selected_norm(UPSTREAM_DIRECTION),
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'matched_upstream_alpha': MATCHED_UPSTREAM_ALPHA,
    'parent_direction_sha256': sha256_file(PARENT_DIRECTION_PATH),
}
write_json_atomic(DIRECTION_LIBRARY_PATH, direction_metadata)
print(direction_metadata)
assert abs(CODE_STEERING_NORM-float(parent_direction['code_norm']))<1e-4, 'Frozen code reconstruction differs; inspect direction_reconstruction_audit.json'


In [ ]:
%%cell_progress 12/15 Freeze reconstructed vectors and input/runtime provenance before interventions
arrays={'upstream':UPSTREAM_DIRECTION.float().numpy().ravel(),
        'semantic':SEMANTIC_DIRECTION.float().numpy().ravel(),
        **{'code_'+a+'_'+b:v.float().numpy().ravel() for (a,b),v in CODE_DIRECTIONS.items()}}
vector_path=OUTPUT_ROOT/'frozen_directions.npz'
if vector_path.is_file():
    with np.load(vector_path,allow_pickle=False) as old:
        assert set(old.files)==set(arrays) and all(np.array_equal(old[k],v) for k,v in arrays.items()), 'Frozen direction reconstruction changed'
else:np.savez_compressed(vector_path,**arrays)
import transformers
runtime=dict(torch=torch.__version__,transformers=transformers.__version__,cuda=torch.version.cuda,dtype=MODEL_DTYPE,
    batch_size=1,checkpoint=MODEL_REVISION,layer_path=LAYER_PATH,
    toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))})
input_files={str(p):file_hash(p) for p in (OLD_ROOT/'frozen_scene_manifest.jsonl',OLD_ROOT/'frozen_match_manifest.jsonl',
    OLD_ROOT/'frozen_protocol.json',PARENT_ROUTE_PATH,PARENT_DIRECTION_PATH,DEPTH_SUPPORT_PATH,AUDIT_OUTCOME_MANIFEST_PATH)}
input_files.update(image_hashes)
input_files.update({str(p):file_hash(p) for p in AUDIT_OUTCOME_SHARD_DIR.glob('outcome_*.pt')})
freeze(OUTPUT_ROOT/'runtime_and_inputs.json',dict(runtime=runtime,inputs=input_files,implementation=IMPLEMENTATION_SHA256,
    directions=file_hash(vector_path),conditions=list(ALL_CONDITIONS)))
# This explicit reference can be used on the separate new shared confirmation cohort.
reference=dict(model='qwen27b',model_id=MODEL_ID,revision=MODEL_REVISION,eligible=True,
    upstream_layer=UPSTREAM_LAYER,downstream_layer=LABEL_LAYER,dose=MATCHED_UPSTREAM_ALPHA,
    directions_sha256=file_hash(vector_path),method='historical_pls4_prototype_frozen_reference',runtime=runtime,
    evidence_role='reference_selected_in_prior_experiments_not_current_development')
freeze(OUTPUT_ROOT/'selected_route.json',reference)


In [ ]:
%%cell_progress 13/15 Validate zero replay and one new removal path, then keep all scientific outcomes
smoke_row=scene_manifest[0];examples,mappings=build_target_examples(smoke_row)
ex=examples['forward'];base,h,m=capture_examples({'x':ex},CAPTURE_LAYERS)
untreated={layer:h['x'][layer][m['x']] for layer in (SEMANTIC_LAYER,LABEL_LAYER)}
ops,axes=build_path_operations(smoke_row,'forward',mappings['forward'],untreated)
zero,_=forward_path_condition(ex,{},untreated,{'condition':'zero'})
assert np.max(np.abs(centered_scores(zero)-centered_scores(base['x'])))<=.15
trial,_=forward_path_condition(ex,ops['upstream_equal_l49_seed0'],untreated,{'condition':'technical_write_check'})
print('One-path numerical check:',trial['write_audits'])
# A discrete norm-matching failure remains an explicit observation, not a relaxed threshold.


In [ ]:
%%cell_progress 14/15 Run all original and additional conditions
cell_stage('Processing independent exact-physics scenes with resumable per-scene shards')

capture_identity = {
    'runtime_and_inputs_sha256': file_hash(OUTPUT_ROOT/'runtime_and_inputs.json'),
    'vector_sha256': file_hash(OUTPUT_ROOT/'frozen_directions.npz'),
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'scene_manifest_sha256': sha256_file(SCENE_MANIFEST_PATH),
    'direction_sha256': sha256_file(DIRECTION_LIBRARY_PATH),
    'core_conditions': list(CORE_CONDITIONS),
    'right_only_conditions': list(RIGHT_ONLY_CONDITIONS),
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

progress = tqdm(scene_manifest, desc='Sequential handoff blocking', unit='scene')
for row in progress:
    shard_path = SHARD_DIR / f"{row['scene_id']}.json"
    expected_conditions = (
        ALL_CONDITIONS if row['side'] == 'right' else CORE_CONDITIONS
    )
    expected_records = len(MAPPING_VARIANTS) * len(expected_conditions)
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        assert len(existing['records']) == expected_records
        continue

    examples, mappings = build_target_examples(row)
    records = []
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    toward_exact_sign = 1.0 if exact == 'left' else -1.0
    for variant in MAPPING_VARIANTS:
        example, mapping = examples[variant], mappings[variant]
        clean_scores, clean_hidden, clean_masks = capture_examples(
            {variant: example}, CAPTURE_LAYERS
        )
        clean = clean_scores[variant]
        clean_margin = semantic_margin(clean, exact, wrong)
        untreated = {
            layer: clean_hidden[variant][layer][clean_masks[variant]]
            for layer in (SEMANTIC_LAYER, LABEL_LAYER)
        }
        operations, axes = build_path_operations(
            row, variant, mapping, untreated
        )
        for condition in expected_conditions:
            progress.set_postfix_str(
                f"{row['cohort']} | {row['side']} | {condition}"
            )
            record, captured = forward_path_condition(
                example, operations[condition], untreated,
                {'condition': condition},
            )
            margin = semantic_margin(record, exact, wrong)
            l49_shift = toward_exact_sign * vector_projection(
                captured[SEMANTIC_LAYER] - untreated[SEMANTIC_LAYER],
                SEMANTIC_DIRECTION,
            )
            l55_shift = vector_projection(
                captured[LABEL_LAYER] - untreated[LABEL_LAYER],
                axes['code'],
            )
            record.update({
                'match_id': row['match_id'], 'pair_id': row['pair_id'],
                'scene_id': row['scene_id'], 'cohort': row['cohort'],
                'side': row['side'], 'baseline_status': row['baseline_status'],
                'style_family': row['style_family'], 'style_id': row['style_id'],
                'mapping_variant': variant,
                'diagnostic_cell': row['diagnostic_cell'],
                'exact_semantic': exact, 'shortcut_semantic': wrong,
                'clean_margin': clean_margin,
                'margin_shift': margin - clean_margin,
                'l49_toward_exact': l49_shift,
                'l55_toward_exact_code': l55_shift,
                'rescued': bool(record['predicted_semantic'] == exact),
                'induced_opposite': bool(record['predicted_semantic'] == wrong),
            })
            records.append(record)
    write_json_atomic(shard_path, {
        'schema_version': 'sequential-handoff-independent-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'scene_id': row['scene_id'], 'records': records,
    })
    progress.set_postfix_str(
        f"{row['cohort']} | {row['side']} | {row['scene_id']}"
    )
    torch.cuda.empty_cache()
progress.close()
print('Completed scene shards:', len(list(SHARD_DIR.glob('*.json'))))


In [ ]:
%%cell_progress 15/15 Analyse direct same-magnitude contrasts without relabelling the old cohort
all_records=[]
for path in sorted(SHARD_DIR.glob('*.json')):
    payload=read_json(path)
    assert payload['capture_identity_sha256']==CAPTURE_SHA256
    all_records.extend(payload['records'])
frame=pd.DataFrame([{k:v for k,v in r.items() if k not in ('candidate_logprobs','candidate_probabilities','write_audits')} for r in all_records])
save_json(ANALYSIS_DIR/'intervention_records.json',all_records)
frame.to_csv(ANALYSIS_DIR/'all_conditions.csv',index=False)
frame[~frame.norm_valid.astype(bool)].to_csv(ANALYSIS_DIR/'norm_match_failures.csv',index=False)
contrasts=[('upstream_l49','upstream_repair_block_l49','upstream_equal_l49'),
           ('upstream_l55','upstream_repair_block_l55','upstream_equal_l55'),
           ('l49_l55','l49_repair_block_l55','l49_equal_l55')]
tests=[];family_tables=[]
for path_name,target,random_prefix in contrasts:
    for group,sub in [('right_error',frame[frame.cohort.eq('right_error')&frame.side.eq('right')]),
                      ('both_correct_right',frame[frame.cohort.eq('both_correct')&frame.side.eq('right')])]:
        selected=sub[sub.condition.eq(target)]
        random=sub[sub.condition.str.startswith(random_prefix+'_seed')]
        if selected.empty:continue
        assert random.groupby(['match_id','scene_id','mapping_variant']).size().eq(4).all()
        for metric in ('margin_shift','rescued'):
            a=selected.groupby('match_id')[metric].mean().reset_index();b=random.groupby('match_id')[metric].mean().reset_index()
            values=paired(b,a,'match_id',metric)
            tests.append(dict(path=path_name,group=group,metric=metric,
                scope='all_attempted',**bootstrap(values,[path_name,group,metric])))
            family_tables.extend(dict(path=path_name,group=group,metric=metric,family_id=fid,difference=float(v)) for fid,v in zip(b.match_id,values))
        valid=set(random.groupby('match_id').norm_valid.all().loc[lambda x:x].index)
        if valid:
            a=selected[selected.match_id.isin(valid)].groupby('match_id').margin_shift.mean().reset_index()
            b=random[random.match_id.isin(valid)].groupby('match_id').margin_shift.mean().reset_index()
            tests.append(dict(path=path_name,group=group,metric='margin_shift',scope='all_four_controls_valid',
                **bootstrap(paired(b,a,'match_id','margin_shift'),[path_name,group,'valid'])))
    # Forward/reverse are compared directly using the same family IDs.
    subset=frame[frame.cohort.eq('right_error')&frame.side.eq('right')]
    mapping_effects=[]
    for variant,sub in subset.groupby('mapping_variant'):
        a=sub[sub.condition.eq(target)].groupby('match_id').margin_shift.mean().reset_index()
        b=sub[sub.condition.str.startswith(random_prefix+'_seed')].groupby('match_id').margin_shift.mean().reset_index()
        if a.empty:continue
        vals=paired(b,a,'match_id','margin_shift')
        mapping_effects.extend(dict(match_id=fid,variant=variant,effect=float(v)) for fid,v in zip(b.match_id,vals))
    pd.DataFrame(mapping_effects).to_csv(ANALYSIS_DIR/f'{path_name}_mapping_effects.csv',index=False)
    if mapping_effects:
        p=pd.DataFrame(mapping_effects).pivot(index='match_id',columns='variant',values='effect')
        result=bootstrap((p.forward-p.reverse).to_numpy(),[path_name,'mapping_interaction'])
        result['p_value']=min(1.,2*min(result['p_value'],1-result['p_value']+1/10001))
        tests.append(dict(path=path_name,group='mapping_difference',metric='margin_shift',scope='secondary',**result))
test_frame=pd.DataFrame(tests);test_frame['q_value']=np.nan
primary=(test_frame.group.eq('right_error')&test_frame.metric.eq('margin_shift')&test_frame.scope.eq('all_attempted'))
assert primary.sum()==3
test_frame.loc[primary,'q_value']=bh(test_frame.loc[primary,'p_value'])
test_frame.to_csv(ANALYSIS_DIR/'same_magnitude_tests.csv',index=False)
pd.DataFrame(family_tables).to_csv(ANALYSIS_DIR/'family_differences.csv',index=False)
late=frame[frame.condition.isin(['l55_repair','l55_bypass_after_l49_block'])&frame.cohort.eq('right_error')&frame.side.eq('right')]
a=late[late.condition.eq('l55_bypass_after_l49_block')].groupby('match_id').margin_shift.mean().reset_index()
b=late[late.condition.eq('l55_repair')].groupby('match_id').margin_shift.mean().reset_index()
save_json(ANALYSIS_DIR/'bypass_vs_late_only.json',bootstrap(paired(a,b,'match_id','margin_shift'),'bypass_vs_late_only'))
display(test_frame);display(frame.groupby(['cohort','side','condition']).rescued.mean().unstack('condition'))
save_json(ANALYSIS_DIR/'RUN_COMPLETE.json',dict(status='complete',historical_families=MATCHED_UNITS,
    records=len(frame),new_family_replication=False,all_norms_valid=bool(frame.norm_valid.all())))
(ANALYSIS_DIR/'REPORT_ZH.md').write_text('# 同幅度随机阻断对照\n\n原 32 家族及所有原条件保留；新增三条路径的四个随机方向。\n'
    '主指标是同幅度随机移除减去指定阻断的答案分数差；正值表示指定阻断造成更大损失。\n'
    '结合实际匹配失败明细、正确率和映射差异解释结果，不自动宣布完整物理计算路径。\n',encoding='utf-8')
print('Complete:',ANALYSIS_DIR)
